In [ ]:
from deepagents import create_deep_agent

In [ ]:
# Setup 1/4: config + secrets
import os
import asyncio
import base64
import json
import re
from dotenv import load_dotenv

load_dotenv(override=True)

MODEL = os.getenv("MODEL", "anthropic:claude-sonnet-5")
BASE = "https://parabank.parasoft.com/parabank"
ALLOWED_HOSTS = {"parabank.parasoft.com"}
SECRETS = {"username": "PARABANK_USERNAME", "password": "PARABANK_PASSWORD"}


def resolve_secret(name: str) -> str:
    """Look up a secret by name. Raises on unknown name or empty value."""
    if name not in SECRETS:
        raise KeyError(f"unknown secret name: {name!r}")
    value = os.environ.get(SECRETS[name], "")
    if not value:
        raise RuntimeError(f"env var {SECRETS[name]} is empty or not set")
    return value


print("model:", MODEL, "| base:", BASE)

In [ ]:
# Setup 2/4: browser
# Open a visible browser on the login page (skips if one is already open in this kernel).
from playwright.async_api import async_playwright

if "page" not in globals():
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=False)
    context = await browser.new_context(viewport={"width": 1280, "height": 900})
    page = await context.new_page()
await page.goto(f"{BASE}/index.htm")
print("opened:", page.url)


In [ ]:
# Setup 3/4: domain guard
from urllib.parse import urlparse


def host_allowed(url: str) -> bool:
    if url == "about:blank":
        return True
    return urlparse(url).hostname in ALLOWED_HOSTS

In [ ]:
# Setup 4/4: numbered screenshot (PlaywrightSurface)
from dataclasses import dataclass

OBSERVE_JS = """
() => {
  document.querySelectorAll('[data-cua-ref]').forEach(e => e.removeAttribute('data-cua-ref'));
  const sel = 'a[href], button, input:not([type=hidden]), select, textarea, [role=button], [role=link], [onclick]';
  const roleOf = (el) => {
    const r = el.getAttribute('role'); if (r) return r;
    const t = el.tagName.toLowerCase();
    if (t === 'a') return 'link';
    if (t === 'button') return 'button';
    if (t === 'select') return 'combobox';
    if (t === 'textarea') return 'textbox';
    if (t === 'input') {
      const ty = (el.getAttribute('type') || 'text').toLowerCase();
      if (['submit', 'button', 'reset', 'image'].includes(ty)) return 'button';
      if (ty === 'checkbox') return 'checkbox';
      if (ty === 'radio') return 'radio';
      return 'textbox';
    }
    return 'generic';
  };
  const nameOf = (el) => {
    const aria = el.getAttribute('aria-label'); if (aria) return aria.trim();
    if (el.labels && el.labels.length) return el.labels[0].innerText.trim();
    const t = el.tagName.toLowerCase();
    const ty = (el.getAttribute('type') || '').toLowerCase();
    if (t === 'input' && ['submit', 'button', 'reset'].includes(ty)) return (el.value || '').trim();
    const txt = (el.innerText || '').trim(); if (txt) return txt.slice(0, 80);
    return (el.getAttribute('placeholder') || el.getAttribute('title') ||
            el.getAttribute('alt') || el.getAttribute('name') || '').trim();
  };
  const items = [];
  document.querySelectorAll(sel).forEach((el) => {
    const r = el.getBoundingClientRect();
    const st = getComputedStyle(el);
    if (r.width < 2 || r.height < 2 || st.visibility === 'hidden' || st.display === 'none') return;
    const ref = items.length + 1;
    el.setAttribute('data-cua-ref', String(ref));
    const valueOf = (el) => {
      const t = el.tagName.toLowerCase();
      if (t === 'select') return el.selectedOptions[0] ? el.selectedOptions[0].text.trim() : '';
      if (t === 'input' || t === 'textarea') {
        const ty = (el.getAttribute('type') || 'text').toLowerCase();
        if (['submit', 'button', 'reset', 'image', 'password'].includes(ty)) return '';
        return (el.value || '').trim();
      }
      return '';
    };
    items.push({
      ref, role: roleOf(el), name: nameOf(el), value: valueOf(el),
      x: r.x, y: r.y, w: r.width, h: r.height,
      inViewport: r.bottom > 0 && r.top < innerHeight && r.right > 0 && r.left < innerWidth,
    });
  });
  return items;
}
"""

DRAW_JS = """
(items) => {
  const box = document.createElement('div');
  box.id = '__cua_overlay';
  box.style.cssText = 'position:fixed;inset:0;pointer-events:none;z-index:2147483647';
  items.filter(i => i.inViewport).forEach(i => {
    const b = document.createElement('div');
    b.style.cssText = `position:fixed;left:${i.x}px;top:${i.y}px;width:${i.w}px;height:${i.h}px;border:2px solid red;box-sizing:border-box`;
    const l = document.createElement('span');
    l.textContent = i.ref;
    l.style.cssText = 'position:absolute;left:0;top:-14px;background:red;color:#fff;font:bold 11px monospace;padding:0 3px';
    b.appendChild(l);
    box.appendChild(b);
  });
  document.body.appendChild(box);
}
"""

CLEAR_JS = "() => { const o = document.getElementById('__cua_overlay'); if (o) o.remove(); }"


def format_elements(elements: list[dict]) -> str:
    lines = []
    for e in elements:
        line = f'[{e["ref"]}] {e["role"]} "{e["name"]}"'
        if e.get("value"):
            line += f' = {e["value"]!r}'      # already-typed content, so the model does not have to guess from pixels
        if e.get("options"):
            line += f' options={e["options"]}'
        if not e["inViewport"]:
            line += " (below the fold)"
        lines.append(line)
    return "\n".join(lines)


@dataclass
class Observation:
    png: bytes
    elements: list[dict]
    url: str
    title: str

    def as_text(self) -> str:
        return f"URL: {self.url}\nTitle: {self.title}\nElements:\n{format_elements(self.elements)}"


class PlaywrightSurface:
    """The only place that touches Playwright. Agent tools talk to this."""

    def __init__(self, page):
        self.page = page
        self.last_elements: list[dict] = []

    async def observe(self) -> Observation:
        elements = await self.page.evaluate(OBSERVE_JS)
        await self.page.evaluate(DRAW_JS, elements)
        png = await self.page.screenshot()
        await self.page.evaluate(CLEAR_JS)
        self.last_elements = elements
        return Observation(png, elements, self.page.url, await self.page.title())

    def name_of(self, ref: int) -> str | None:
        for e in self.last_elements:
            if e["ref"] == ref:
                return e["name"]
        return None

In [ ]:
# STEP 1: scanner patch (dropdown options + submit flag). Run right after the surface cell.
OBSERVE_JS = OBSERVE_JS.replace(
    "inViewport: r.bottom",
    "options: el.tagName === 'SELECT' ? Array.from(el.options).map(o => o.text.trim()).slice(0, 15) : null,\n"
    "      submit: (el.tagName === 'BUTTON' && el.type !== 'button') || (el.tagName === 'INPUT' && ['submit', 'image'].includes(el.type)),\n"
    "      inViewport: r.bottom",
)
assert "options:" in OBSERVE_JS and "submit:" in OBSERVE_JS, "patch did not apply"


def format_elements(elements: list[dict]) -> str:
    lines = []
    for e in elements:
        line = f'[{e["ref"]}] {e["role"]} "{e["name"]}"'
        if e.get("options"):
            line += f' options={e["options"]}'
        if not e["inViewport"]:
            line += " (below the fold)"
        lines.append(line)
    return "\n".join(lines)


In [ ]:
# STEP 2: banner, takeover, Approve / Reject / Take over buttons, hand_off
import asyncio
from langgraph.types import Command

HANDBACK = {"event": None}

# ---- Approve / Reject / Take over buttons (their own floating bar, always clickable) ----
DECISION_JS = """(info) => new Promise(resolve => {
  const bar = document.createElement('div');
  bar.style.cssText = 'position:fixed;top:0;left:0;right:0;z-index:2147483647;background:#111;color:#fff;font:14px sans-serif;padding:10px;display:flex;flex-direction:column;gap:8px;align-items:center';
  const t = document.createElement('b'); t.textContent = info.title;
  const d = document.createElement('div'); d.textContent = info.details;
  const row = document.createElement('div'); row.style.cssText = 'display:flex;gap:12px';
  [['Approve', 'a'], ['Reject', 'r'], ['Take over', 't']].forEach(([label, val]) => {
    const b = document.createElement('button'); b.textContent = label;
    b.style.cssText = 'padding:6px 14px;font-size:14px;cursor:pointer';
    b.onclick = () => { bar.remove(); resolve(val); };
    row.appendChild(b);
  });
  bar.append(t, d, row);
  document.body.appendChild(bar);
})"""

# ---- Whole-page lock: while it is the agent's turn, a real human can neither click nor type
# anywhere on the page. Our own injected UI (this banner, the decision bar above) always sits
# on a higher z-index, so it stays usable regardless of lock state. Our OWN automated actions
# use force=True (STEP 3), which bypasses this overlay entirely -- it only stops a real human's
# mouse and keyboard, never Playwright's own dispatched actions. ----
LOCK_JS = """
() => {
  if (!document.getElementById('__cua_lock')) {
    const d = document.createElement('div');
    d.id = '__cua_lock';
    d.style.cssText = 'position:fixed;inset:0;z-index:2147483000;background:transparent;';
    document.documentElement.appendChild(d);
  }
  if (document.activeElement && document.activeElement.blur) document.activeElement.blur();
  if (!window.__cuaBlockKeys) {
    // Let typing through for an element explicitly poked open by RESTRICT_JS (allow_refs mode).
    // Without this check, the click-blocking overlay and the keyboard block were two separate
    // mechanisms: raising an element's z-index let a click focus it, but every keystroke was
    // still swallowed here, at the document level, regardless of what had focus.
    window.__cuaBlockKeys = (e) => {
      if (e.target && e.target.classList && e.target.classList.contains('__cua_allowed')) return;
      e.preventDefault(); e.stopPropagation();
    };
    document.addEventListener('keydown', window.__cuaBlockKeys, true);
    document.addEventListener('keypress', window.__cuaBlockKeys, true);
  }
}
"""
UNLOCK_JS = """
() => {
  document.getElementById('__cua_lock')?.remove();
  if (window.__cuaBlockKeys) {
    document.removeEventListener('keydown', window.__cuaBlockKeys, true);
    document.removeEventListener('keypress', window.__cuaBlockKeys, true);
    window.__cuaBlockKeys = null;
  }
}
"""

# ---- Risky-element block: separate from the general lock above. During a GENERAL takeover
# (filling in a field), the rest of the page opens up but the risky button must still stay
# non-interactive. Only the take-over-to-submit case turns this off too. ----
BLOCK_JS = """(refs) => {
  document.querySelectorAll('.__cua_blocked').forEach(el => {
    el.classList.remove('__cua_blocked');
    el.style.filter = ''; el.style.pointerEvents = ''; el.style.opacity = '';
  });
  refs.forEach(ref => {
    const el = document.querySelector(`[data-cua-ref="${ref}"]`);
    if (el) { el.classList.add('__cua_blocked'); el.style.filter = 'blur(3px)'; el.style.pointerEvents = 'none'; el.style.opacity = '0.5'; }
  });
}"""
UNBLOCK_JS = "() => document.querySelectorAll('.__cua_blocked').forEach(el => { el.classList.remove('__cua_blocked'); el.style.filter = ''; el.style.pointerEvents = ''; el.style.opacity = ''; })"

# ---- Restrict to specific fields: keep the general lock fully ACTIVE, and poke a hole (raise
# z-index above the lock, but below our own UI) for ONLY the given refs. A visible green outline
# shows the human exactly what is expected. Used by request_value/request_missing_values, where
# we know precisely which field(s) are needed -- unlike ask_human or the approval take-over,
# which genuinely need broader access. ----
RESTRICT_JS = """(refs) => {
  document.querySelectorAll('.__cua_allowed').forEach(el => {
    el.classList.remove('__cua_allowed');
    el.style.position = ''; el.style.zIndex = ''; el.style.outline = '';
  });
  refs.forEach(ref => {
    const el = document.querySelector(`[data-cua-ref="${ref}"]`);
    if (el) {
      el.classList.add('__cua_allowed');
      if (getComputedStyle(el).position === 'static') el.style.position = 'relative';
      el.style.zIndex = '2147483001';
      el.style.outline = '3px solid #2a7';
    }
  });
}"""
UNRESTRICT_JS = "() => document.querySelectorAll('.__cua_allowed').forEach(el => { el.classList.remove('__cua_allowed'); el.style.position = ''; el.style.zIndex = ''; el.style.outline = ''; })"

BANNER_ONLY_JS = """
() => {
  if (document.getElementById('__cua_banner')) return;
  const q = sessionStorage.getItem('cua_question') || '';
  const b = document.createElement('div');
  b.id = '__cua_banner';
  b.style.cssText = 'position:fixed;top:0;left:0;right:0;z-index:2147483647;background:#c00;color:#fff;font:14px sans-serif;padding:8px;display:flex;gap:12px;align-items:center;justify-content:center';
  const label = document.createElement('b'); label.textContent = 'YOU are in control.';
  const msg = document.createElement('span'); msg.textContent = q ? 'Agent asks: ' + q : 'Do the step in this page.';
  const btn = document.createElement('button'); btn.textContent = 'Done, hand back to agent';
  btn.style.cssText = 'padding:6px 14px;font-size:14px;cursor:pointer';
  btn.onclick = () => window.__cua_handback();
  b.append(label, msg, btn);
  document.documentElement.appendChild(b);
}
"""

# ---- "You are in control" bar with a Done button. Runs on every page load (add_init_script);
# also applied immediately to the current page. Locked/unlocked state is driven by the SAME
# 'cua_takeover' sessionStorage flag, so both stay in sync across navigations for free. ----
SYNC_UI_JS = """
(() => {
  const active = sessionStorage.getItem('cua_takeover') === '1';
  if (!active) {
    if (!document.getElementById('__cua_lock')) {
      const d = document.createElement('div');
      d.id = '__cua_lock';
      d.style.cssText = 'position:fixed;inset:0;z-index:2147483000;background:transparent;';
      document.documentElement.appendChild(d);
    }
    if (!window.__cuaBlockKeys) {
      window.__cuaBlockKeys = (e) => {
        if (e.target && e.target.classList && e.target.classList.contains('__cua_allowed')) return;
        e.preventDefault(); e.stopPropagation();
      };
      document.addEventListener('keydown', window.__cuaBlockKeys, true);
      document.addEventListener('keypress', window.__cuaBlockKeys, true);
    }
    document.getElementById('__cua_banner')?.remove();
    return;
  }
  document.getElementById('__cua_lock')?.remove();
  if (window.__cuaBlockKeys) {
    document.removeEventListener('keydown', window.__cuaBlockKeys, true);
    document.removeEventListener('keypress', window.__cuaBlockKeys, true);
    window.__cuaBlockKeys = null;
  }
  if (document.getElementById('__cua_banner')) return;
  const q = sessionStorage.getItem('cua_question') || '';
  const b = document.createElement('div');
  b.id = '__cua_banner';
  b.style.cssText = 'position:fixed;top:0;left:0;right:0;z-index:2147483647;background:#c00;color:#fff;font:14px sans-serif;padding:8px;display:flex;gap:12px;align-items:center;justify-content:center';
  const label = document.createElement('b');
  label.textContent = 'YOU are in control.';
  const msg = document.createElement('span');
  msg.textContent = q ? 'Agent asks: ' + q : 'Do the step in this page.';
  const btn = document.createElement('button');
  btn.textContent = 'Done, hand back to agent';
  btn.style.cssText = 'padding:6px 14px;font-size:14px;cursor:pointer';
  btn.onclick = () => window.__cua_handback();
  b.append(label, msg, btn);
  document.documentElement.appendChild(b);
})();
"""

if "_takeover_ready" not in globals():
    async def _handback():
        if HANDBACK["event"]:
            HANDBACK["event"].set()
    await page.expose_function("__cua_handback", _handback)   # page -> notebook signal
    await page.add_init_script(SYNC_UI_JS)                    # keeps lock/banner correct after every navigation
    # sessionStorage lives in the BROWSER TAB, not the Python kernel: restarting the kernel does
    # NOT clear it, since Playwright's browser process is separate and keeps running. Force-clear
    # any flag left over from an earlier, possibly-interrupted run, so a fresh kernel always starts
    # from a known "not in takeover" state regardless of the tab's history.
    await page.evaluate("() => { sessionStorage.removeItem('cua_takeover'); sessionStorage.removeItem('cua_question'); }")
    await page.evaluate(SYNC_UI_JS)                            # apply to the current page right now (locked by default)
    _lock_present = await page.evaluate("() => !!document.getElementById('__cua_lock')")
    print(f"[lock] initial setup done | __cua_lock present on page: {_lock_present} | url={page.url}")
    _takeover_ready = True


async def human_takeover(question: str = "", auto_on_navigate: bool = False, block_risky: bool = True, allow_refs: list[int] | None = None) -> str:
    """Give the live browser to a human. Returns when they click 'Done', OR, if auto_on_navigate
    is set, as soon as the page reloads at all (same url or not -- a form POST-back that
    re-renders the same url still counts).

    allow_refs, when given, is the STRICTEST mode: the general lock stays fully ACTIVE, and only
    these specific elements are poked open (a visible green outline marks them). Everything else
    on the page, including navigation links, stays locked. Used by request_value and
    request_missing_values, which know exactly which field(s) are needed.

    Without allow_refs, the whole page unlocks instead. block_risky=True (the default) then
    ADDITIONALLY blurs and disables every button needs_human (STEP 3) would flag, so a general
    takeover (ask_human) can still not be used to bypass the approval step. The one call site
    that should pass block_risky=False is the take-over-to-submit case in click(), since acting
    on that specific button is the entire point of that one takeover.

    auto_on_navigate=True is for that same approval-step takeover: the human took over
    specifically to click a submit button, so ANY resulting page reload IS the "done" signal.
    """
    HANDBACK["event"] = asyncio.Event()
    visited = []
    print(f"[takeover] started | auto_on_navigate={auto_on_navigate} | block_risky={block_risky} | allow_refs={allow_refs} | url={page.url}")

    def on_nav(frame):
        if frame != page.main_frame:
            return
        visited.append(frame.url)
        print(f"[takeover] framenavigated fired -> {frame.url}")
        if auto_on_navigate and not HANDBACK["event"].is_set():
            HANDBACK["event"].set()
            print("[takeover] auto hand-back triggered")

    page.on("framenavigated", on_nav)
    await page.evaluate("([q]) => { sessionStorage.setItem('cua_takeover', '1'); sessionStorage.setItem('cua_question', q); }", [question])
    if allow_refs is not None:
        await page.evaluate(BANNER_ONLY_JS)     # lock stays active; only these refs are poked open
        await page.evaluate(RESTRICT_JS, allow_refs)
        risky = []
    else:
        await page.evaluate(SYNC_UI_JS)
        risky = [e["ref"] for e in surface.last_elements if e["inViewport"] and needs_human(e["ref"])] if block_risky else []
        if block_risky:
            await page.evaluate(BLOCK_JS, risky)
    _lock_present = await page.evaluate("() => !!document.getElementById('__cua_lock')")
    print(f"[lock] takeover open | __cua_lock present={_lock_present} (True only expected in allow_refs mode) | risky refs blocked: {risky}")
    await HANDBACK["event"].wait()
    print(f"[takeover] hand-back received | visited={visited} | url now={page.url}")
    page.remove_listener("framenavigated", on_nav)
    if allow_refs is not None:
        await page.evaluate(UNRESTRICT_JS)
    elif block_risky:
        await page.evaluate(UNBLOCK_JS)
    try:
        await page.evaluate("sessionStorage.removeItem('cua_takeover'); sessionStorage.removeItem('cua_question');")
        await page.evaluate(SYNC_UI_JS)   # re-lock and remove the banner right away (safe in both modes)
        _lock_present = await page.evaluate("() => !!document.getElementById('__cua_lock')")
        print(f"[lock] takeover ended, re-locked | __cua_lock present (should be True): {_lock_present}")
    except Exception as exc:
        print(f"[lock] could not confirm re-lock after hand-back (page likely moved on): {type(exc).__name__}")
    text = (await page.inner_text("body"))[:300].replace("\n", " ")
    return f"Pages the human visited: {visited or 'none'}. Page now: {page.url}. Page text: {text}"


def approval_info(args: dict) -> dict:
    name = surface.name_of(args["ref"])
    fields = "; ".join(f"{k}: {v}" for k, v in TYPED.items()) or "(nothing typed)"
    return {"title": f"Agent wants to click '{name}' on {page.url.split('/')[-1]}", "details": f"Values it entered: {fields}"}


In [ ]:
# STEP 3: browser tools
import asyncio, base64, functools
from langchain.tools import tool
from langchain.agents.middleware import AgentMiddleware, AgentState, Runtime

ACT_LOCK = asyncio.Lock()


def one_at_a_time(fn):
    @functools.wraps(fn)
    async def wrapper(*a, **k):
        async with ACT_LOCK:
            return await fn(*a, **k)
    return wrapper


TYPED: dict[str, str] = {}      # what the agent entered, by field name (shown in the approval bar)
GIVEN = {"text": ""}            # what the user actually gave us; set before each run
RESULT: dict = {}
DENY_LINKS = ("register", "lookup", "admin")
SENSITIVE_WORDS = ("ssn", "password", "social")


async def _unlocked(coro):
    """Run one Playwright action with the general lock (STEP 2) removed for just that instant.
    force=True alone does NOT bypass the lock: Playwright still dispatches the click/fill by
    coordinate ('use page.mouse over the center of the element', per its own docs), so the real
    browser hit-tests normally and the lock overlay -- being on top -- would receive it instead
    of our intended target. Genuinely removing the lock, only for this one action, is what
    actually works."""
    await page.evaluate(UNLOCK_JS)
    try:
        return await coro
    finally:
        await page.evaluate(LOCK_JS)


async def _click(self, ref: int):
    await _unlocked(self.page.locator(f'[data-cua-ref="{ref}"]').click(timeout=5000, force=True))
    await self.page.wait_for_timeout(600)
    try:
        await self.page.wait_for_load_state("load", timeout=5000)
    except Exception:
        pass

async def _type_text(self, ref: int, text: str):
    await _unlocked(self.page.locator(f'[data-cua-ref="{ref}"]').fill(text, timeout=5000, force=True))

async def _page_text(self) -> str:
    return (await self.page.inner_text("body"))[:4000]

PlaywrightSurface.click = _click
PlaywrightSurface.type_text = _type_text
PlaywrightSurface.page_text = _page_text
surface = PlaywrightSurface(page)


def _blocks(prefix: str, obs) -> list[dict]:
    """Package text and a screenshot as one tool result the model can read."""
    return [
        {"type": "text", "text": f"{prefix}\n{obs.as_text()}"},
        {"type": "image", "base64": base64.b64encode(obs.png).decode(), "mime_type": "image/png"},
    ]


def _describe(ref: int, hint: str = "") -> str:
    """Best available description of an element: the code's own name, the model's visual
    hint, both together, or a last-resort 'field N' only if neither is available. This is
    what closes the gap between D2 (the model can read a label visually, from the screenshot)
    and a human-facing message (which used to ask only the code, never the model)."""
    name = (surface.name_of(ref) or "").strip()
    hint = (hint or "").strip()
    if name and hint:
        return f"{name} ({hint})"
    return name or hint or f"field {ref}"


START_PAGES = {"overview.htm", "index.htm"}   # start pages: asking a human here is too early. Open the task page first.


def current_page() -> str:
    """Page name from the URL: no query, no ;jsessionid=, no trailing slash, lower case."""
    return page.url.split("?")[0].split(";")[0].rstrip("/").rsplit("/", 1)[-1].lower()


async def current_value(ref: int) -> str:
    """Read whatever is currently in this field, live from the page. Empty string if none or unreadable."""
    try:
        loc = page.locator(f'[data-cua-ref="{ref}"]')
        tag = await loc.evaluate("el => el.tagName.toLowerCase()")
        if tag == "select":
            return (await loc.evaluate("el => el.selectedOptions[0] ? el.selectedOptions[0].text : ''")).strip()
        return (await loc.input_value(timeout=1000)).strip()
    except Exception:
        return ""


async def _ask_for_value(ref: int, field: str, kind: str) -> list:
    """The agent tried to enter a value the user never gave. Hand the browser to a human --
    unless the field already has a value, in which case refuse and say so. This is a code
    guarantee, not something left to the model's memory of what it already asked about."""
    val = await current_value(ref)
    if val:
        return _blocks(
            f"SKIP: '{field}' already has a value ({val!r}). Do not ask about it again; move to a different field.",
            await surface.observe(),
        )
    report = await human_takeover(
        f"I need a value for '{field}' and you did not give me one. Please {kind} it yourself in the page, then click Done.",
        allow_refs=[ref],
    )
    return _blocks(
        f"A human entered the value for '{field}' themselves. Do NOT type it again. {report}",
        await surface.observe(),
    )


@tool(parse_docstring=True)
@one_at_a_time
async def observe() -> list:
    """Look at the current page.

    Returns:
        A screenshot with red numbered boxes, plus a text list of the numbered elements.
    """
    return _blocks("Current page.", await surface.observe())


SAFE_SUBMITS = {"log in", "find transactions"}   # buttons that do not change data
AUTO_LIMIT = None                                # None = always ask a human. Later: 500.0 for small transfers
DECLINED: set[str] = set()                       # buttons a human refused; never clicked again this run


def _amount() -> float:
    raw = TYPED.get("amount", "").replace("$", "").replace(",", "")
    try:
        return float(raw)
    except ValueError:
        return float("inf")                      # unknown amount counts as risky


def needs_human(ref: int) -> bool:
    """Every button except a short safe list needs a human. Links (navigation) run freely."""
    el = next((e for e in surface.last_elements if e["ref"] == ref), None)
    role = (el or {}).get("role")
    name = ((el or {}).get("name") or "").strip().lower()
    risky = bool(el) and (bool(el.get("submit")) or role == "button") and name not in SAFE_SUBMITS
    if risky and "transfer" in name and AUTO_LIMIT is not None and _amount() <= AUTO_LIMIT:
        risky = False
    print(f"approval check -> role={role!r} name={name!r} risky={risky}")
    return risky


LOGIN_ATTEMPTS = {"count": 0}
LOGIN_BLOCKED = {"blocked": False}
# Best-guess wording for ParaBank's own login failure message; not verified against the live
# site in this session. The 3-attempt hard cap below is the guaranteed backstop regardless of
# whether this text matches -- it does not depend on guessing the wording right.
LOGIN_FAILURE_TEXTS = ("could not be verified", "user does not exist", "invalid username or password")
LOGIN_ATTEMPT_LIMIT = 3


def login_check(attempts_so_far: int, page_text_after: str, limit: int = LOGIN_ATTEMPT_LIMIT):
    """Pure decision: after a login click, should further attempts be blocked, and why."""
    attempts = attempts_so_far + 1
    failed_text = next((t for t in LOGIN_FAILURE_TEXTS if t in page_text_after.lower()), None)
    if failed_text:
        return attempts, True, f"the login page reported: '{failed_text}'"
    if attempts >= limit:
        return attempts, True, f"login was attempted {attempts} times with no success"
    return attempts, False, None


@tool(parse_docstring=True)
@one_at_a_time
async def click(ref: int) -> list:
    """Click an element on the page.

    Buttons that change data ask a human for approval first. You do not need to do anything for that.

    Args:
        ref: Number of the element in the latest screenshot and list.

    Returns:
        The new page state. If a human declined, the result says DECLINED and you must stop.
    """
    name = (surface.name_of(ref) or "").strip().lower()
    if any(w in name for w in DENY_LINKS):
        return _blocks(f"DENIED: '{name}' is not allowed.", await surface.observe())
    if name in DECLINED:
        return _blocks("DECLINED earlier by a human. Do not retry. Call finish with 'DECLINED:' and stop.", await surface.observe())
    if name == "log in" and LOGIN_BLOCKED["blocked"]:
        return _blocks("BLOCKED: login already failed or hit its attempt limit. Do not try again. Call finish with a summary starting 'STUCK:' and stop.", await surface.observe())
    if needs_human(ref):
        # The page is already locked (STEP 2) whenever it is not an active takeover, so a
        # human cannot click the real button while this decision is pending -- only our own
        # decision bar (its own higher z-index) is interactive right now.
        choice = await page.evaluate(DECISION_JS, approval_info({"ref": ref}))
        if choice == "r":
            DECLINED.add(name)
            return _blocks("DECLINED by a human. Do not retry or work around it. Call finish with 'DECLINED:' and stop.", await surface.observe())
        if choice == "t":
            report = await human_takeover(question="", auto_on_navigate=True, block_risky=False)   # this IS the approval step; the human may act on the button itself
            return _blocks(f"A human completed this step manually in the browser. {report} Do NOT click again. Check the result from the page text, then call finish.", await surface.observe())
        # choice == "a": approved, fall through. Our own click below uses force=True (STEP 3),
        # so the still-locked page does not stop it.
    try:
        await surface.click(ref)
    except Exception as exc:
        return _blocks(f"CLICK FAILED for [{ref}]: {type(exc).__name__}", await surface.observe())
    if not host_allowed(page.url):
        await page.go_back()
        return _blocks("BLOCKED: left the allowed site. Went back.", await surface.observe())
    if name == "log in":
        page_text_after = await surface.page_text()
        LOGIN_ATTEMPTS["count"], blocked, reason = login_check(LOGIN_ATTEMPTS["count"], page_text_after)
        if blocked:
            LOGIN_BLOCKED["blocked"] = True
            return _blocks(
                f"STOP: login failed ({reason}). Do not try again. Call finish with a summary starting 'STUCK:' explaining this.",
                await surface.observe(),
            )
    return _blocks(f"Clicked [{ref}].", await surface.observe())


@tool(parse_docstring=True)
@one_at_a_time
async def type_text(ref: int, text: str) -> list:
    """Type normal text into an input box.

    Only type values the user gave you in the goal. Never invent a value.

    Args:
        ref: Number of the input box in the latest screenshot and list.
        text: The text to type. It must come from the user's goal.

    Returns:
        The new page state.
    """
    field = _describe(ref)
    if any(w in field.lower() for w in SENSITIVE_WORDS) or text.strip().lower() not in GIVEN["text"].lower():
        return await _ask_for_value(ref, field, "type")
    try:
        await surface.type_text(ref, text)
    except Exception as exc:
        return _blocks(f"TYPE FAILED for [{ref}]: {type(exc).__name__}", await surface.observe())
    TYPED[field.lower()] = text
    return _blocks(f"Typed into [{ref}].", await surface.observe())


@tool(parse_docstring=True)
@one_at_a_time
async def type_secret(ref: int, name: str) -> list:
    """Type a stored secret into an input box, without ever seeing its value.

    Use this to log in. You give only the secret's name.

    Args:
        ref: Number of the input box in the latest screenshot and list.
        name: Secret name. Either 'username' or 'password'.

    Returns:
        The new page state. The value is never included.
    """
    if name not in SECRETS:
        return _blocks(f"UNKNOWN SECRET '{name}'. Use one of: {list(SECRETS)}", await surface.observe())
    if not host_allowed(page.url):
        return _blocks("REFUSED: this site is not on the allowlist.", await surface.observe())
    try:
        await _unlocked(page.locator(f'[data-cua-ref="{ref}"]').fill(resolve_secret(name), timeout=5000, force=True))
    except Exception as exc:
        return _blocks(f"FAILED for [{ref}]: {type(exc).__name__}", await surface.observe())
    return _blocks(f"Typed secret '{name}' into [{ref}].", await surface.observe())


@tool(parse_docstring=True)
@one_at_a_time
async def select_option(ref: int, option: str) -> list:
    """Choose an option in a dropdown.

    Only choose options the user named in the goal. Never guess.

    Args:
        ref: Number of the dropdown in the latest screenshot and list.
        option: Visible text of the option, exactly as shown in its options list.

    Returns:
        The new page state.
    """
    field = _describe(ref)
    if option.strip().lower() not in GIVEN["text"].lower():
        return await _ask_for_value(ref, field, "choose")
    try:
        await _unlocked(page.locator(f'[data-cua-ref="{ref}"]').select_option(label=option, timeout=5000, force=True))
    except Exception as exc:
        return _blocks(f"SELECT FAILED for [{ref}]: {type(exc).__name__}", await surface.observe())
    TYPED[field.lower()] = option
    return _blocks(f"Selected '{option}' in [{ref}].", await surface.observe())


@tool(parse_docstring=True)
@one_at_a_time
async def page_text() -> str:
    """Read the visible text of the whole page (balances, tables, messages).

    Returns:
        The page text, up to 4000 characters.
    """
    return await surface.page_text()


@tool(parse_docstring=True)
@one_at_a_time
async def ask_human(question: str) -> list:
    """Ask a human for help by handing over the browser.

    Use this when you are unsure which element to pick, the page looks unexpected,
    a step failed twice, or you would have to guess. Never guess account choices or amounts.

    Args:
        question: What you are unsure about and what you need the human to do.

    Returns:
        What the human did, and the new page state.
    """
    print(f"guard (ask_human) -> page={current_page()!r} start_page={current_page() in START_PAGES}")
    if current_page() in START_PAGES:
        return _blocks(
            "NOT YET: you are still on the start page. Open the page where this task is done first "
            "(use the menu). Then, with the form on screen, use request_value on each field you cannot fill.",
            await surface.observe(),
        )
    report = await human_takeover(question)
    return _blocks(f"A human took over and handed back. {report}", await surface.observe())


@tool(parse_docstring=True)
@one_at_a_time
async def request_value(ref: int, hint: str) -> list:
    """Ask the human to fill one field that you cannot fill, because the user did not give the value.

    Open the page that has the field first. Then point at the field, and say what you see: read
    the label from the screenshot even if the element has no name in the code. This happens on
    older pages where a label sits in a nearby table cell or heading, not attached to the input.
    Your visual reading is what the human sees in the request, so give your best reading even if
    you are not fully sure. The page scrolls to the field, the human types the value there and
    hands control back. Do not type into that field yourself afterwards.

    Args:
        ref: Number of the field (input box or dropdown) in the latest screenshot and list.
        hint: What you see as this field's label or purpose, read from the screenshot.

    Returns:
        The new page state, after the human is done.
    """
    print(f"guard (request_value) -> page={current_page()!r} start_page={current_page() in START_PAGES}")
    if current_page() in START_PAGES:
        return _blocks(
            "NOT YET: you are still on the start page. Open the page that has this form first.",
            await surface.observe(),
        )
    field = _describe(ref, hint)
    try:
        loc = page.locator(f'[data-cua-ref="{ref}"]')
        await loc.scroll_into_view_if_needed(timeout=3000)
        await loc.focus(timeout=3000)
    except Exception:
        pass                                     # the human can still find it; do not fail the request
    return await _ask_for_value(ref, field, "enter")


def missing_field_labels(elements: list[dict], hints: dict[str, str] | None = None) -> list[tuple[int, str]]:
    """Empty, visible, fillable fields, in page order. Pure: no browser, no network call."""
    hints = hints or {}
    out = []
    for e in elements:
        if not e["inViewport"] or e["role"] not in ("textbox", "combobox") or e.get("value"):
            continue
        ref = e["ref"]
        name = (e.get("name") or "").strip()
        hint = hints.get(str(ref), "").strip()
        label = f"{name} ({hint})" if name and hint else (name or hint or f"field {ref}")
        out.append((ref, label))
    return out


@tool(parse_docstring=True)
@one_at_a_time
async def request_missing_values(hints: dict[str, str] = {}) -> list:
    """Ask a human to fill every empty field on the current page in one go.

    Use this once you have typed or chosen every value the user actually gave you, and the
    form still has empty fields left. Call this ONCE for all of them, rather than calling
    request_value field by field. If some fields are still empty after the human clicks Done,
    call this again: only the fields still empty will be shown, not the ones already filled.

    Args:
        hints: Optional. Maps a field's ref number (as text, e.g. "17") to your own reading of
            its label, for fields with no clear name in the code.

    Returns:
        The new page state, after the human is done.
    """
    if current_page() in START_PAGES:
        return _blocks("NOT YET: open the page that has this form first.", await surface.observe())
    missing = missing_field_labels(surface.last_elements, hints)
    if not missing:
        return _blocks("Nothing is missing right now.", await surface.observe())
    labels = [label for _, label in missing]
    question = "Please fill in these fields, then click Done: " + "; ".join(labels)
    report = await human_takeover(question, allow_refs=[ref for ref, _ in missing])
    return _blocks(f"A human filled in what they chose to. {report}", await surface.observe())


@tool(parse_docstring=True)
async def finish(summary: str, values: dict[str, str]) -> str:
    """Report the final result and stop.

    If you cannot make progress, start the summary with 'STUCK:'. If a human declined, start it with 'DECLINED:'.

    Args:
        summary: One-line summary of what you did or why you stopped.
        values: The requested facts, for example {"account_id": "13344", "balance": "$100.00"}.

    Returns:
        A confirmation. Stop after this.
    """
    RESULT.clear()
    RESULT.update({"summary": summary, "values": values})
    return "Recorded. Stop now."


BROWSER_TOOLS = [observe, click, type_text, type_secret, select_option, page_text, request_value, request_missing_values, ask_human, finish]
print("tools ready:", [t.name for t in BROWSER_TOOLS])


In [ ]:
# (left here on purpose, not deleted) "JEV_API_KEY" was a wrong guess.
# The real name is TYPESAFE_API_KEY. "jev-latest" is only TypeSafe's own
# model name for their classifier, not an env var. Fixed in STEP 4 below.


In [ ]:
# STEP 3d: TypeSafe tool selection (Choice). Classifies "what job is this step" and
# narrows the tool list to that job. Off by default (needs TYPESAFE_API_KEY): this is
# the only tool-selection help in this notebook now (the free page-based triage was removed).
NEVER_HIDE = {"observe", "click", "type_secret", "finish"}   # always allowed, whatever the job

JOB_EXTRA_TOOLS = {
    "login":      {"type_secret"},
    "fill_form":  {"type_text", "select_option"},
    "read_value": {"page_text"},
    "need_human": {"request_value", "ask_human"},
}
JOB_CRITERIA = {
    "login": "The page shows a username or password field, or we have not logged in yet.",
    "fill_form": "A form is on screen and a field still needs a value typed or a dropdown chosen.",
    "read_value": "We need to read a value already on the page, such as a balance or a confirmation message.",
    "need_human": "We are unsure which element to use, or a value we need was not given by the user.",
}
JOB_CONFIDENCE_THRESHOLD = 0.8


def job_tool_names(job: str) -> set[str]:
    """Tools this job needs, plus the always-allowed set. Pure: no network, no LLM."""
    return NEVER_HIDE | JOB_EXTRA_TOOLS.get(job, set())


def confidence_gate(base_tools: set[str], job: str, confidence: float,
                     threshold: float = JOB_CONFIDENCE_THRESHOLD) -> set[str]:
    """Narrow base_tools to this job's tools, but only if the classifier is confident.
    Below the threshold, fail OPEN: return base_tools unchanged rather than guess wrong."""
    if confidence < threshold:
        return base_tools
    return base_tools & job_tool_names(job)


class TypeSafeToolRouterMiddleware(AgentMiddleware):
    """Classifies the step's job with TypeSafe's Choice primitive and narrows the tool
    list to it. Sends the current page path and the last tool result's text to
    api.typesafe.ai (D50/D52 caveat: never enable on a run that may show real account
    data). Any error here (network, auth, timeout) fails OPEN: the request goes through
    unmodified."""

    def __init__(self, classifier):
        self.classifier = classifier

    async def awrap_model_call(self, request, handler):
        try:
            state = f"page={current_page()!r}. last result: {str(request.messages[-1].content)[:400]!r}"
            response = await self.classifier.ainvoke(
                {"state": state, "questions": {"job": Choice(instructions="What kind of step is this?", criteria=JOB_CRITERIA)}}
            )
            answer = response.choices["job"]
            named = {t.name for t in request.tools if hasattr(t, "name")}
            keep = confidence_gate(named, answer.choice, answer.confidence)
            request = request.override(tools=[t for t in request.tools if not hasattr(t, "name") or t.name in keep])
            print(f"typesafe job -> {answer.choice!r} confidence={answer.confidence:.2f} kept={sorted(keep)}")
        except Exception as exc:
            print(f"typesafe job router FAILED, continuing with no change: {type(exc).__name__}: {exc}")
        return await handler(request)


In [ ]:
# # STEP 3e: offline checks for the job mapping and the confidence gate (no network, no key)
# _ALL = {"observe", "click", "type_text", "type_secret", "select_option",
#         "page_text", "request_value", "ask_human", "finish"}

# assert confidence_gate(_ALL, "login", 0.9) == NEVER_HIDE | {"type_secret"}
# assert confidence_gate(_ALL, "fill_form", 0.75) == NEVER_HIDE | {"type_text", "select_option"}
# assert confidence_gate(_ALL, "read_value", 0.8) == NEVER_HIDE | {"page_text"}
# assert confidence_gate(_ALL, "need_human", 0.99) == NEVER_HIDE | {"request_value", "ask_human"}
# assert confidence_gate(_ALL, "login", 0.59) == _ALL                          # low confidence: fail open
# assert confidence_gate(_ALL, "login", JOB_CONFIDENCE_THRESHOLD) == NEVER_HIDE | {"type_secret"}  # boundary is inclusive
# _from_triage = _ALL - {"request_value", "ask_human"}
# assert NEVER_HIDE <= confidence_gate(_from_triage, "fill_form", 0.9)          # never-hide survives even when the base tool set is already reduced
# assert confidence_gate(_ALL, "not_a_real_job", 0.95) == NEVER_HIDE           # unknown job: no crash, no extras
# print("typesafe job-router checks passed")


In [ ]:
# STEP 4: system prompt and agents (use agent_hitl for anything that changes data)
from langgraph.checkpoint.memory import MemorySaver

SYSTEM_PROMPT = """You are an expert browser operator. You drive a real browser on a banking demo site, and you can search the web for outside facts.

## Browser tools
Every tool result shows a screenshot with red numbered boxes plus a list of numbered elements. A field already filled shows its current value, e.g. [7] textbox "City" = '2'. Never ask about a field that already shows a value; move to one that does not. Dropdowns list their options. Refer to elements only by number. Numbers change after every action, so use the latest list.
- observe: look at the page.
- click(ref), type_text(ref, text), select_option(ref, option): act on elements.
- type_secret(ref, name): type a stored secret ('username' or 'password'). You never see the value.
- page_text: read the visible page text (balances, messages).
- request_value(ref, hint): ONE field you cannot fill. Prefer request_missing_values instead when several fields are empty.
- request_missing_values(hints): every empty field on the current page, asked in ONE go, not one at a time. hints maps a ref number to your own label reading, for fields with no name in the code.
- ask_human(question): only when you are unsure what to click. Not for missing values.
- finish(summary, values): report the result, logout and then stop.

## How to work
1. Call observe first. If you see a login form, log in with type_secret, then confirm the account overview appears.
2. Do the task by the shortest path. Read values with page_text and copy them exactly. Never invent a value.
3. Use ONLY values the user gave you in the goal. Never make one up. If values you need (payee, amount, account, address) are missing: FIRST open the page where the task is done (use the menu links), THEN call request_missing_values ONCE to ask for everything still empty at the same time. Never ask a human while you are still on the start page.
4. If a tool says a human entered a value, do not type it again. Continue with the next step.
5. Buttons that change data (Send Payment, Transfer, Open New Account, and so on) need a human. Click the button when the form is ready; the system asks the human for approval by itself. If the result says DECLINED, never retry or work around it: call finish with 'DECLINED:'.
6. Make ONE tool call at a time. Do not click into a field before typing.
7. Stay on the banking site. If you are lost or repeat the same action 3 times, call finish with 'STUCK:' and say why.
8. Attempt login at most 3 times. If the page says the login could not be verified, stop immediately -- do not retry. Call finish with a summary starting 'STUCK:' explaining what happened.
9. Do not use ls, read_file, write_file, edit_file, delete, glob, grep or task.
9. all steps must end with logout. STUCK or happy path or DECLINED, always logout after you finish. Do not leave the session open.
"""

internet_search = {"type": "web_search_20260209", "name": "web_search"}

# Optional: route each step to Haiku or Sonnet, via TypeSafe (a third-party classifier service).
# Off by default. Turns on only if TYPESAFE_API_KEY is set in .env (get one at typesafe.ai).
# Trade-off, decided and recorded in DECISIONS.md (D50): step text is sent to api.typesafe.ai
# to pick the model. Never enable this for a run that may show real account data.
TYPESAFE_API_KEY = os.getenv("TYPESAFE_API_KEY", "")
HAIKU_MODEL = "anthropic:claude-haiku-4-5-20251001"
SONNET_MODEL = "anthropic:claude-sonnet-5"

middleware = []   # no key: no tool-selection help (only TypeSafe provides it, see STEP 3d)
if TYPESAFE_API_KEY:
    from langchain_typesafe import Choice, TypeSafeClassifier
    from langchain_typesafe.experimental.middleware import ModelChoice, ModelRouterMiddleware

    router = ModelRouterMiddleware(
        choices={
            "fast": ModelChoice(
                model=HAIKU_MODEL,
                criteria="A single simple step: reading the page, or one obvious click, type, or select with no ambiguity.",
            ),
            "powerful": ModelChoice(
                model=SONNET_MODEL,
                criteria="Anything else: planning, choosing between several similar elements, forms, or any step before a risky click.",
            ),
        },
        instructions="Pick the cheapest model that can do the step correctly. If unsure, pick 'powerful'.",
    )
    middleware = [TypeSafeToolRouterMiddleware(TypeSafeClassifier()), router]
    print(f"model router ON (TypeSafe): fast={HAIKU_MODEL} | powerful={SONNET_MODEL}")
else:
    print("model router OFF: no TYPESAFE_API_KEY in .env. Using MODEL only:", MODEL)

agent = create_deep_agent(
    model=MODEL,
    tools=[internet_search, *BROWSER_TOOLS],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=MemorySaver(),
    middleware=middleware,
)

agent_hitl = agent      # same agent: the approval now lives inside the click tool
print("agent ready | approval is enforced inside the click tool")


In [ ]:
# Run a bill payment. Every value below is given by the user, so the agent may type it.
import uuid

GOAL = ("transfer from mine to 999999")   # <-- change 14232 to YOUR account id
GIVEN["text"] = GOAL
TYPED.clear()
await page.goto(f"{BASE}/overview.htm")
cfg = {"configurable": {"thread_id": f"bill-{uuid.uuid4().hex[:6]}"}, "recursion_limit": 100}
out = await agent.ainvoke({"messages": [{"role": "user", "content": GOAL}]}, config=cfg)
print(out["messages"][-1].content)
